# English Whisper transcription — one ZIP at a time

Run the cells in order. For every ZIP archive, run the final cell once, upload one ZIP, and download its processed result.

In [ ]:
!pip install -q openai-whisper
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
import torch
import whisper

MODEL_NAME = 'base'  # tiny, base, small, medium, large
LANGUAGE = 'en'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}. Loading {MODEL_NAME} model...')
model = whisper.load_model(MODEL_NAME, device=device)

In [ ]:
# Run this cell once for EACH ZIP file. Upload only one ZIP when asked.
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import files

AUDIO_EXTENSIONS = {'.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.webm'}

print('Upload one ZIP file, for example lists_one.zip.')
uploaded = files.upload()
zip_files = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
if len(zip_files) != 1:
    raise ValueError('Please upload exactly one ZIP file, then run this cell again.')

zip_path = zip_files[0]
input_root = Path('/content') / f'{zip_path.stem}_input'
output_root = Path('/content') / f'{zip_path.stem}_whisper_output'
shutil.rmtree(input_root, ignore_errors=True)
shutil.rmtree(output_root, ignore_errors=True)
input_root.mkdir(parents=True)
output_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(input_root)

audio_files = sorted(
    path for path in input_root.rglob('*')
    if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
)
if not audio_files:
    raise FileNotFoundError('No supported audio files were found in this ZIP file.')

for audio_path in audio_files:
    relative_path = audio_path.relative_to(input_root)
    lesson_folder = output_root / relative_path.with_suffix('')
    lesson_folder.mkdir(parents=True, exist_ok=True)
    print(f'\nTranscribing: {relative_path}')
    shutil.copy2(audio_path, lesson_folder / f'audio{audio_path.suffix.lower()}')

    result = model.transcribe(str(audio_path), language=LANGUAGE, fp16=(device == 'cuda'))
    timestamps = [
        {'start': item['start'], 'end': item['end'], 'text': item['text'].strip()}
        for item in result['segments']
    ]
    transcript = '\n'.join(item['text'] for item in timestamps)
    (lesson_folder / 'raw_timestamp.json').write_text(
        json.dumps(timestamps, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    (lesson_folder / 'raw_text.txt').write_text(transcript, encoding='utf-8')
    (lesson_folder / 'text.txt').write_text(transcript, encoding='utf-8')

# First archive: transcripts and timestamps only (no audio files).
text_only_zip = Path('/content') / f'{zip_path.stem}_whisper_text_only.zip'
with zipfile.ZipFile(text_only_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_root.rglob('*'):
        if file_path.is_file() and not file_path.name.startswith('audio.'):
            archive.write(file_path, file_path.relative_to(output_root))


print(f'\nFinished. Downloading {text_only_zip.name} first...')
files.download(str(text_only_zip))


In [ ]:

# Second archive: the same output plus the copied audio files.
full_zip = shutil.make_archive(
    str(Path('/content') / f'{zip_path.stem}_whisper_with_audio'),
    'zip',
    output_root,
)
print(f'Downloading {Path(full_zip).name} next...')
files.download(full_zip)